In [0]:
# ====================================================================
# GOLD LAYER - CUSTOMER COHORT ANALYSIS
# ====================================================================
# Purpose: Track customer retention by signup month cohort
# ====================================================================

from pyspark.sql.functions import (
    col, min, max, count, countDistinct, datediff,
    date_trunc, months_between, current_timestamp, round
)
from pyspark.sql.window import Window

# Configuration
PROJECT_NAME = "retail"
CATALOG = "workspace"
SILVER_SCHEMA = f"{PROJECT_NAME}_silver"
GOLD_SCHEMA = f"{PROJECT_NAME}_gold"

print("=" * 80)
print("👥 GOLD LAYER - CUSTOMER COHORT ANALYSIS")
print("=" * 80)
print(f"Source: {CATALOG}.{SILVER_SCHEMA}.silver_customers_master")
print(f"Target: {CATALOG}.{GOLD_SCHEMA}.gold_customer_cohorts")
print("=" * 80 + "\n")

In [0]:
customers = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_customers_master")
enriched_orders = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.silver_orders_enriched")

print(f"✅ Loaded customers: {customers.count():,}")
print(f"✅ Loaded orders: {enriched_orders.count():,}")
print("=" * 80 + "\n")

In [0]:
# ====================================================================
# RETENTION ANALYSIS
# ====================================================================
print("📈 Calculating retention rates...\n")

# For each order, identify which cohort the customer belongs to
orders_with_cohort = (enriched_orders
    .filter(col("order_status") == "delivered")
    .join(
        customer_cohorts.select("customer_id", "cohort_month"),
        "customer_id",
        "inner"
    )
    .withColumn("order_month", date_trunc("month", col("order_purchase_timestamp")))
    .withColumn(
        "months_since_cohort",
        months_between(col("order_month"), col("cohort_month")).cast("int")
    )
)

# Calculate retention by cohort and month
retention_rates = (orders_with_cohort
    .groupBy("cohort_month", "months_since_cohort")
    .agg(
        countDistinct("customer_id").alias("active_customers"),
        countDistinct("order_id").alias("orders"),
        sum("item_total_value").alias("revenue")
    )
    .withColumn("_created_at", current_timestamp())
    .orderBy("cohort_month", "months_since_cohort")
)

print(f"✅ Retention data points: {retention_rates.count():,}")
print("=" * 80 + "\n")

In [0]:
retention_table = f"{CATALOG}.{GOLD_SCHEMA}.gold_customer_retention"

(retention_rates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(retention_table))

print(f"✅ Written: {retention_table}")
print("=" * 80 + "\n")

In [0]:
%sql
-- Retention heatmap (first 12 months)
SELECT 
    cohort_month,
    months_since_cohort,
    active_customers,
    orders,
    ROUND(revenue, 2) as revenue
FROM workspace.retail_gold.gold_customer_retention
WHERE months_since_cohort <= 12
ORDER BY cohort_month, months_since_cohort;